# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [7]:
print("""
ML Task Type: Ranking / Scoring

Why: We are scoring and ranking content items to prioritize which ones need to be refreshed first. The task consists of predicting a priority score for each content item, representing the urgency and potential impact of a content update. In practice, the business operates under a capacity limit (e.g. content editors can review only the top-K pages per week). Hence, the ultimate output is a ranked queue where the highest-ranked pages are presented to the editor first. This is framed as a ranking task evaluated by Precision@K (precision of the top-K recommendations) rather than simple classification accuracy.
""")


ML Task Type: Ranking / Scoring

Why: We are scoring and ranking content items to prioritize which ones need to be refreshed first. The task consists of predicting a priority score for each content item, representing the urgency and potential impact of a content update. In practice, the business operates under a capacity limit (e.g. content editors can review only the top-K pages per week). Hence, the ultimate output is a ranked queue where the highest-ranked pages are presented to the editor first. This is framed as a ranking task evaluated by Precision@K (precision of the top-K recommendations) rather than simple classification accuracy.



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [8]:
print("""
What we predict (Target): We predict whether a page is likely to experience traffic decay in a future observation window.

Where the label comes from:
- Starter dataset (Proxy): In this 30k teachable slice, we use `is_declining_label`, which is computed as `trend_direction == "down"`. This is a proxy label because it is computed over the current window rather than a separate future window. However, it is still an observed outcome in the search console data (derived from trailing 30d impressions vs. preceding 30d impressions), not a product-defined rule. This prevents the circular result trap of learning our own rules.
- Capstone / Warehouse (Ideal Target): For the full warehouse release, we define the target over a distinct future window (e.g., whether the page's search clicks or impressions drop by >20% in the 30 days *following* the 90-day feature aggregation period). This preserves temporal separation and prevents feature leakage.
""")


What we predict (Target): We predict whether a page is likely to experience traffic decay in a future observation window.

Where the label comes from:
- Starter dataset (Proxy): In this 30k teachable slice, we use `is_declining_label`, which is computed as `trend_direction == "down"`. This is a proxy label because it is computed over the current window rather than a separate future window. However, it is still an observed outcome in the search console data (derived from trailing 30d impressions vs. preceding 30d impressions), not a product-defined rule. This prevents the circular result trap of learning our own rules.
- Capstone / Warehouse (Ideal Target): For the full warehouse release, we define the target over a distinct future window (e.g., whether the page's search clicks or impressions drop by >20% in the 30 days *following* the 90-day feature aggregation period). This preserves temporal separation and prevents feature leakage.



## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [9]:
print("""
Success Metric: Precision@K (specifically Precision@50) and Average Precision (AP)

Justification: The business problem is to rank pages for a human content review team with limited capacity (e.g. 50 pages a week). The model's accuracy on pages ranked low in the queue (e.g., page 20,000) does not matter. What matters is that the pages ranked in the top-K are indeed true decline risks. For our starter validation, we use `Precision@50` (of the top 50 pages predicted as decline risks, what fraction actually declined) and `Average Precision` (overall ranking quality across the precision-recall curve).

What 'good' looks like:
- The current product's fixed rule baseline gets a Precision@50 of 0.24 (only 12 of the top 50 pages are actual declines).
- A 'good' trained model should significantly exceed this baseline. Our Random Forest starter model achieves a Precision@50 of 0.74 (37 of the top 50 pages are actual declines), which is a 3x improvement in review efficiency.
""")


Success Metric: Precision@K (specifically Precision@50) and Average Precision (AP)

Justification: The business problem is to rank pages for a human content review team with limited capacity (e.g. 50 pages a week). The model's accuracy on pages ranked low in the queue (e.g., page 20,000) does not matter. What matters is that the pages ranked in the top-K are indeed true decline risks. For our starter validation, we use `Precision@50` (of the top 50 pages predicted as decline risks, what fraction actually declined) and `Average Precision` (overall ranking quality across the precision-recall curve).

What 'good' looks like:
- The current product's fixed rule baseline gets a Precision@50 of 0.24 (only 12 of the top 50 pages are actual declines).
- A 'good' trained model should significantly exceed this baseline. Our Random Forest starter model achieves a Precision@50 of 0.74 (37 of the top 50 pages are actual declines), which is a 3x improvement in review efficiency.



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
!rm -rf ml
!git clone https://github.com/AsmSafone/ML-Internship ml

Cloning into 'ml'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 103 (delta 22), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 1.84 MiB | 10.71 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [11]:
import pandas as pd
import numpy as np

# Load the starter CSV
df = pd.read_csv('/content/ml/data/raw/content_refresh_anonymized.csv')

# Compute target proxy label
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Select representative columns to show unit of analysis and target
show_cols = [
    'content_id',
    'client_id',
    'content_age_days',
    'impressions_90d',
    'ctr',
    'avg_position',
    'trend_direction',
    'is_declining_label'
]

# Print dataframe shape
print(f"Dataframe shape: {df.shape}")
print("\nUnit of Analysis (one row = one content item per client):")
print(df[show_cols].head().to_string())

Dataframe shape: (30000, 45)

Unit of Analysis (one row = one content item per client):
             content_id          client_id  content_age_days  impressions_90d   ctr  avg_position trend_direction  is_declining_label
0  content_304f48230142  client_f369cb89fc               187             3803  0.76          10.6            down                   1
1  content_a1fb4e703a9e  client_4e07408562               445            15320  0.05          20.3            down                   1
2  content_9aa793d4d895  client_7f2253d7e2               141            12581  0.09          36.5            down                   1
3  content_331d6c4de07b  client_19581e27de               463            11751  0.49           6.2          stable                   0
4  content_d99b7a2d90ca  client_3fdba35f04               263            19140  0.13          44.0            down                   1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [12]:
print("""
Why ML beats a fixed rule:
1. Multivariate interaction: Traffic decay is not caused by a single signal but by the interaction of multiple factors (e.g., older pages that maintain position but drop in engagement/scroll rate, or high-impression pages with decaying CTR and rising GSC position). An if-statement cannot easily combine 10+ shifting rates and counts without becoming unmaintainable.
2. Brittle static thresholds: Static cutoffs (e.g., `content_age_days > 180`) are arbitrary. A page in a high-competition space might decay in 90 days, while a page in a stable niche is fine for 360 days. ML models can handle non-linear thresholds and scale them based on other context.
3. Unbalanced baseline across clients: Different clients have different tracking histories, baseline traffic levels, and keyword difficulty. A static rule behaves differently on each client. A trained ML model can learn client-relative patterns and generalize across client-holdout splits without hardcoding.
""")


Why ML beats a fixed rule:
1. Multivariate interaction: Traffic decay is not caused by a single signal but by the interaction of multiple factors (e.g., older pages that maintain position but drop in engagement/scroll rate, or high-impression pages with decaying CTR and rising GSC position). An if-statement cannot easily combine 10+ shifting rates and counts without becoming unmaintainable.
2. Brittle static thresholds: Static cutoffs (e.g., `content_age_days > 180`) are arbitrary. A page in a high-competition space might decay in 90 days, while a page in a stable niche is fine for 360 days. ML models can handle non-linear thresholds and scale them based on other context.
3. Unbalanced baseline across clients: Different clients have different tracking histories, baseline traffic levels, and keyword difficulty. A static rule behaves differently on each client. A trained ML model can learn client-relative patterns and generalize across client-holdout splits without hardcoding.



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.